# Level 2: Data Processing & Feature Engineering

## Project Objective

The objective of Level 2 is to process the train schedule data and create useful features for further analysis. This includes standardizing schedule time fields, calculating journey duration, classifying routes based on distance, and generating station-wise train frequency.

In [1]:
import pandas as pd
import numpy as np

In [2]:
df = pd.read_csv("../DATASET/Dataset1.csv")

In [3]:
df.head()

,SN,Train_No,Station_Code,1A,2A,3A,SL,Station_Name,Route_Number,Arrival_time,Departure_Time,Distance
0,1,107,SWV,100,100,100,100,SAWANTWADI R,1,00:00:00,10:25:00,0
1,2,107,THVM,260,228,196,164,THIVIM,1,11:06:00,11:08:00,32
2,3,107,KRMI,345,296,247,198,KARMALI,1,11:28:00,11:30:00,49
3,4,107,MAO,490,412,334,256,MADGOAN JN.,1,12:10:00,00:00:00,78
4,1,108,MAO,100,100,100,100,MADGOAN JN.,1,00:00:00,20:30:00,0


In [4]:
df[["Train_No", "Station_Name", "Arrival_time", "Departure_Time"]].head(10)

,Train_No,Station_Name,Arrival_time,Departure_Time
0,107,SAWANTWADI R,00:00:00,10:25:00
1,107,THIVIM,11:06:00,11:08:00
2,107,KARMALI,11:28:00,11:30:00
3,107,MADGOAN JN.,12:10:00,00:00:00
4,108,MADGOAN JN.,00:00:00,20:30:00
5,108,KARMALI,21:04:00,21:06:00
6,108,THIVIM,21:26:00,21:28:00
7,108,SAWANTWADI R,22:25:00,00:00:00
8,128,MADGOAN JN.,19:40:00,19:40:00
9,128,KARMALI,20:18:00,20:20:00


In [5]:
df[["Arrival_time", "Departure_Time"]].dtypes

Arrival_time      str
Departure_Time    str
dtype: object

## Standardize Schedule Time Fields

Arrival and departure time fields are converted into a consistent datetime/time representation to support journey duration calculations and further time-based analysis.

In [6]:
print(df["Arrival_time"].head(10))
print(df["Departure_Time"].head(10))

0    00:00:00
1    11:06:00
2    11:28:00
3    12:10:00
4    00:00:00
5    21:04:00
6    21:26:00
7    22:25:00
8    19:40:00
9    20:18:00
Name: Arrival_time, dtype: str
0    10:25:00
1    11:08:00
2    11:30:00
3    00:00:00
4    20:30:00
5    21:06:00
6    21:28:00
7    00:00:00
8    19:40:00
9    20:20:00
Name: Departure_Time, dtype: str


In [7]:
print(df["Arrival_time"].dtype)
print(df["Departure_Time"].dtype)

str
str


In [8]:
df["Arrival_time"] = pd.to_datetime(df["Arrival_time"], format="%H:%M:%S")
df["Departure_Time"] = pd.to_datetime(df["Departure_Time"], format="%H:%M:%S")

In [9]:
df[["Arrival_time", "Departure_Time"]].dtypes

Arrival_time      datetime64[us]
Departure_Time    datetime64[us]
dtype: object

In [10]:
df[df["Train_No"] == 107][
    ["Train_No", "SN", "Station_Name", "Arrival_time", "Departure_Time"]
]

,Train_No,SN,Station_Name,Arrival_time,Departure_Time
0,107,1,SAWANTWADI R,1900-01-01 00:00:00,1900-01-01 10:25:00
1,107,2,THIVIM,1900-01-01 11:06:00,1900-01-01 11:08:00
2,107,3,KARMALI,1900-01-01 11:28:00,1900-01-01 11:30:00
3,107,4,MADGOAN JN.,1900-01-01 12:10:00,1900-01-01 00:00:00


In [11]:
print("Arrival 00:00:", (df["Arrival_time"].dt.time == pd.Timestamp("00:00:00").time()).sum())
print("Departure 00:00:", (df["Departure_Time"].dt.time == pd.Timestamp("00:00:00").time()).sum())

Arrival 00:00: 2003
Departure 00:00: 1970


In [12]:
df[(df["Arrival_time"].dt.time == pd.Timestamp("00:00:00").time()) |
   (df["Departure_Time"].dt.time == pd.Timestamp("00:00:00").time())][
    ["Train_No", "SN", "Station_Name", "Arrival_time", "Departure_Time"]
].head(20)

,Train_No,SN,Station_Name,Arrival_time,Departure_Time
0,107,1,SAWANTWADI R,1900-01-01 00:00:00,1900-01-01 10:25:00
3,107,4,MADGOAN JN.,1900-01-01 12:10:00,1900-01-01 00:00:00
4,108,1,MADGOAN JN.,1900-01-01 00:00:00,1900-01-01 20:30:00
7,108,4,SAWANTWADI R,1900-01-01 22:25:00,1900-01-01 00:00:00
990,4831,1,JODHPUR JN.,1900-01-01 00:00:00,1900-01-01 13:30:00
1011,4831,22,HARIDWAR JN,1900-01-01 10:45:00,1900-01-01 00:00:00
1328,5715,1,NEW JALPAIGU,1900-01-01 00:00:00,1900-01-01 03:00:00
1329,5715,2,KISHANGANJ,1900-01-01 05:10:00,1900-01-01 00:00:00
1330,5716,1,KISHANGANJ,1900-01-01 00:00:00,1900-01-01 04:00:00
1331,5716,2,NEW JALPAIGU,1900-01-01 06:30:00,1900-01-01 00:00:00


In [13]:
df = df.sort_values(["Train_No", "SN"]).reset_index(drop=True)

In [14]:
journey_summary = df.groupby("Train_No").agg(
    Start_Time=("Departure_Time", "first"),
    End_Time=("Arrival_time", "last")
).reset_index()

journey_summary["Journey_Duration"] = (
    journey_summary["End_Time"] - journey_summary["Start_Time"]
)

journey_summary.loc[
    journey_summary["Journey_Duration"] < pd.Timedelta(0),
    "Journey_Duration"
] += pd.Timedelta(days=1)

In [15]:
journey_summary["Journey_Duration_Hours"] = (
    journey_summary["Journey_Duration"].dt.total_seconds() / 3600
)

In [16]:
journey_summary[
    ["Train_No", "Journey_Duration", "Journey_Duration_Hours"]
].head(10)

,Train_No,Journey_Duration,Journey_Duration_Hours
0,107,0 days 01:45:00,1.750000
1,108,0 days 01:55:00,1.916667
2,128,0 days 22:05:00,22.083333
3,290,0 days 08:00:00,8.000000
4,401,0 days 12:30:00,12.500000
5,421,0 days 09:00:00,9.000000
6,422,0 days 02:45:00,2.750000
7,477,0 days 03:10:00,3.166667
8,502,0 days 23:00:00,23.000000
9,504,0 days 01:00:00,1.000000


In [17]:
df = df.merge(
    journey_summary[
        ["Train_No", "Journey_Duration", "Journey_Duration_Hours"]
    ],
    on="Train_No",
    how="left"
)

In [18]:
df[
    ["Train_No", "Station_Name",
     "Journey_Duration", "Journey_Duration_Hours"]
].head(10)

,Train_No,Station_Name,Journey_Duration,Journey_Duration_Hours
0,107,SAWANTWADI R,0 days 01:45:00,1.750000
1,107,THIVIM,0 days 01:45:00,1.750000
2,107,KARMALI,0 days 01:45:00,1.750000
3,107,MADGOAN JN.,0 days 01:45:00,1.750000
4,108,MADGOAN JN.,0 days 01:55:00,1.916667
5,108,KARMALI,0 days 01:55:00,1.916667
6,108,THIVIM,0 days 01:55:00,1.916667
7,108,SAWANTWADI R,0 days 01:55:00,1.916667
8,128,MADGOAN JN.,0 days 22:05:00,22.083333
9,128,KARMALI,0 days 22:05:00,22.083333


In [19]:
journey_summary["Journey_Duration_Hours"].describe()

count    11113.000000
mean         4.600225
std          5.352782
min          0.000000
25%          1.016667
50%          2.200000
75%          6.083333
max         23.916667
Name: Journey_Duration_Hours, dtype: float64

In [20]:
journey_summary["Route_Type"] = pd.cut(
    journey_summary["Journey_Duration_Hours"],
    bins=[0, 1.016667, 6.083333, float("inf")],
    labels=["Short", "Medium", "Long"]
)

journey_summary["Route_Type"] = (
    journey_summary["Route_Type"]
    .cat.add_categories(["Unknown"])
)

journey_summary.loc[
    journey_summary["Journey_Duration_Hours"] == 0,
    "Route_Type"
] = "Unknown"

In [21]:
journey_summary[
    ["Train_No", "Journey_Duration_Hours", "Route_Type"]
].head(20)

,Train_No,Journey_Duration_Hours,Route_Type
0,107,1.750000,Medium
1,108,1.916667,Medium
2,128,22.083333,Long
3,290,8.000000,Long
4,401,12.500000,Long
5,421,9.000000,Long
6,422,2.750000,Medium
7,477,3.166667,Medium
8,502,23.000000,Long
9,504,1.000000,Short


In [24]:
journey_summary["Route_Type"].value_counts()

Route_Type
Medium     5537
Short      2787
Long       2783
Unknown       6
Name: count, dtype: int64

In [25]:
df = df.merge(
    journey_summary[["Train_No", "Route_Type"]],
    on="Train_No",
    how="left"
)

In [26]:
df[
    ["Train_No", "Station_Name",
     "Journey_Duration_Hours", "Route_Type"]
].head(15)

,Train_No,Station_Name,Journey_Duration_Hours,Route_Type
0,107,SAWANTWADI R,1.750000,Medium
1,107,THIVIM,1.750000,Medium
2,107,KARMALI,1.750000,Medium
3,107,MADGOAN JN.,1.750000,Medium
4,108,MADGOAN JN.,1.916667,Medium
5,108,KARMALI,1.916667,Medium
6,108,THIVIM,1.916667,Medium
7,108,SAWANTWADI R,1.916667,Medium
8,128,MADGOAN JN.,22.083333,Long
9,128,KARMALI,22.083333,Long


In [27]:
journey_summary["Route_Type"].value_counts()

Route_Type
Medium     5537
Short      2787
Long       2783
Unknown       6
Name: count, dtype: int64

In [28]:
journey_summary["Route_Type"].value_counts(normalize=True) * 100

Route_Type
Medium     49.824530
Short      25.078737
Long       25.042743
Unknown     0.053991
Name: proportion, dtype: float64

In [29]:
journey_summary[
    journey_summary["Journey_Duration_Hours"] == 0
]

,Train_No,Start_Time,End_Time,Journey_Duration,Journey_Duration_Hours,Route_Type
991,12617,1900-01-01 13:15:00,1900-01-01 13:15:00,0 days,0.0,Unknown
1205,12851,1900-01-01 08:55:00,1900-01-01 08:55:00,0 days,0.0,Unknown
2022,16318,1900-01-01 21:55:00,1900-01-01 21:55:00,0 days,0.0,Unknown
2327,18233,1900-01-01 17:15:00,1900-01-01 17:15:00,0 days,0.0,Unknown
2381,18477,1900-01-01 20:55:00,1900-01-01 20:55:00,0 days,0.0,Unknown
2804,22633,1900-01-01 14:15:00,1900-01-01 14:15:00,0 days,0.0,Unknown


In [30]:
zero_duration_trains = journey_summary.loc[
    journey_summary["Journey_Duration_Hours"] == 0,
    "Train_No"
]

df[df["Train_No"].isin(zero_duration_trains)].groupby("Train_No").agg(
    Station_Count=("Station_Name", "count"),
    First_Station=("Station_Name", "first"),
    Last_Station=("Station_Name", "last")
).reset_index()

,Train_No,Station_Count,First_Station,Last_Station
0,12617,48,ERNAKULAM. J,HAZRAT NIZAM
1,12851,17,BILASPUR JN.,CHENNAI CENT
2,16318,75,SHRI MATA VA,KANNIYAKUMAR
3,18233,72,INDORE BG,BILASPUR JN.
4,18477,76,PURI,HARIDWAR JN
5,22633,29,TRIVANDRUM C,HAZRAT NIZAM


In [31]:
station_frequency = df.groupby("Station_Name")["Train_No"].nunique().reset_index()

station_frequency.columns = [
    "Station_Name",
    "Train_Frequency"
]

station_frequency = station_frequency.sort_values(
    "Train_Frequency",
    ascending=False
).reset_index(drop=True)

station_frequency.head(20)

,Station_Name,Train_Frequency
0,CST-MUMBAI,1027
1,KALYAN JN,828
2,THANE,796
3,SEALDAH,745
4,CHENNAI BEAC,738
5,HOWRAH JN.,699
6,DADAR,598
7,DUM DUM JN.,463
8,KURLA,462
9,TAMBARAM,434


# Level 2 — Key Findings & Observations

## Key Findings

1. **Time Standardization**
   - Arrival and departure time fields were converted into a consistent datetime format to support reliable time-based calculations.
   - The dataset contains `00:00:00` values at certain journey boundaries, such as the arrival time of the first station and the departure time of the last station.
   - These values were handled based on their position in the train's route rather than treating every `00:00:00` value as missing data.

2. **Journey Duration**
   - Total journey duration was calculated for each train using the first departure time and the final arrival time.
   - Overnight journeys were handled by adding one day when the calculated end time was earlier than the start time.
   - A numerical `Journey_Duration_Hours` feature was created for further analysis and route classification.
   - The calculated journey durations range from 0 to approximately 23.92 hours.

3. **Route Classification**
   - Trains were classified into Short, Medium, and Long routes based on journey duration.
   - The final classification contains:
     - Medium: 5,537 trains
     - Short: 2,787 trains
     - Long: 2,783 trains
     - Unknown: 6 trains
   - Medium-duration routes are the most common category in the processed dataset.
   - Six trains resulted in a journey duration of zero hours. Further investigation showed that these trains contain multiple stations, indicating that the zero duration is associated with the available schedule time values rather than the absence of a route. These records were therefore classified as `Unknown`.

4. **Station-wise Train Frequency**
   - Train frequency was calculated for each station using the number of unique trains operating through the station.
   - `CST-MUMBAI` has the highest train frequency with 1,027 unique trains.
   - Other highly frequent stations include `KALYAN JN`, `THANE`, `SEALDAH`, and `CHENNAI BEAC`.

5. **Feature Engineering**
   - The processing stage generated the following features for further analysis and Power BI visualization:
     - `Journey_Duration`
     - `Journey_Duration_Hours`
     - `Route_Type`
     - `Train_Frequency`

6. **Data Quality Observation**
   - The `00:00:00` values require contextual interpretation because some represent boundary events in a train's schedule rather than conventional missing values.
   - Validation of zero-duration trains helped identify potential schedule-data limitations before using the engineered features for further analysis.